# IPL Auction Price Analysis: Descriptive Statistics

This notebook provides exploratory analysis of IPL auction prices and player performance data (2008-2025).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

df = pd.read_csv('../data/analysis/auction_inflation_adjusted.csv')
print(f"Loaded {len(df)} records")
df.head()

## 1. Price Trends Over Time

In [ ]:
yearly = df.groupby('year').agg({
    'price_nominal_cr': ['sum', 'mean', 'max'],
    'price_2024_cr': ['sum', 'mean', 'max'],
    'player_name': 'count'
}).round(2)
yearly.columns = ['total_nominal', 'avg_nominal', 'max_nominal', 
                  'total_real', 'avg_real', 'max_real', 'n_players']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(yearly.index, yearly['total_nominal'], 'o-', label='Nominal ₹', color='blue')
ax1.plot(yearly.index, yearly['total_real'], 's--', label='Real ₹ (2024)', color='green')
ax1.set_xlabel('Year')
ax1.set_ylabel('Total Spending (₹ Crore)')
ax1.set_title('Total Auction Spending by Year')
ax1.legend()
ax1.set_xticks(yearly.index)
ax1.tick_params(axis='x', rotation=45)

ax2 = axes[1]
ax2.plot(yearly.index, yearly['avg_nominal'], 'o-', label='Nominal ₹', color='blue')
ax2.plot(yearly.index, yearly['avg_real'], 's--', label='Real ₹ (2024)', color='green')
ax2.set_xlabel('Year')
ax2.set_ylabel('Average Price (₹ Crore)')
ax2.set_title('Average Player Price by Year')
ax2.legend()
ax2.set_xticks(yearly.index)
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/analysis/fig_price_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Price Distribution by Year

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

df_box = df[df['price_2024_cr'].notna()].copy()
df_box = df_box[df_box['price_2024_cr'] > 0]

years = sorted(df_box['year'].unique())
data = [df_box[df_box['year'] == y]['price_2024_cr'].values for y in years]

bp = ax.boxplot(data, positions=range(len(years)), widths=0.6, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')

ax.set_xticks(range(len(years)))
ax.set_xticklabels(years, rotation=45)
ax.set_xlabel('Year')
ax.set_ylabel('Price (₹ Crore, 2024 terms)')
ax.set_title('Distribution of Auction Prices by Year (Inflation-Adjusted)')

plt.tight_layout()
plt.savefig('../data/analysis/fig_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Top 10 Deals by Year

In [ ]:
top_by_year = df.loc[df.groupby('year')['price_2024_cr'].idxmax()]
top_by_year = top_by_year.sort_values('year')

fig, ax = plt.subplots(figsize=(14, 6))

bars = ax.bar(top_by_year['year'], top_by_year['price_2024_cr'], color='steelblue', edgecolor='black')

for bar, (_, row) in zip(bars, top_by_year.iterrows()):
    name = row['player_name'].split()[-1] if pd.notna(row['player_name']) else ''
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
            name, ha='center', va='bottom', fontsize=8, rotation=45)

ax.set_xlabel('Year')
ax.set_ylabel('Price (₹ Crore, 2024 terms)')
ax.set_title('Highest-Paid Player Each Year (Inflation-Adjusted)')
ax.set_xticks(top_by_year['year'])
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/analysis/fig_top_deals.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop deal each year:")
print(top_by_year[['year', 'player_name', 'team_x', 'price_nominal_cr', 'price_2024_cr']].to_string(index=False))

## 4. Price by Player Role

In [ ]:
role_stats = df.groupby('role').agg({
    'price_2024_cr': ['mean', 'median', 'max', 'count']
}).round(2)
role_stats.columns = ['mean', 'median', 'max', 'count']
role_stats = role_stats.sort_values('mean', ascending=False)
print("Price statistics by role:")
print(role_stats)

fig, ax = plt.subplots(figsize=(10, 6))

roles_to_plot = ['All-Rounder', 'Batsman', 'Bowler', 'Wicket-Keeper']
df_roles = df[df['role'].isin(roles_to_plot) & df['price_2024_cr'].notna()]

sns.boxplot(data=df_roles, x='role', y='price_2024_cr', order=roles_to_plot, ax=ax)
ax.set_xlabel('Role')
ax.set_ylabel('Price (₹ Crore, 2024 terms)')
ax.set_title('Auction Prices by Player Role')

plt.tight_layout()
plt.savefig('../data/analysis/fig_price_by_role.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Indian vs Overseas Premium

In [ ]:
nat_yearly = df.groupby(['year', 'nationality']).agg({
    'price_2024_cr': 'mean',
    'player_name': 'count'
}).reset_index()
nat_yearly.columns = ['year', 'nationality', 'avg_price', 'n_players']

nat_pivot = nat_yearly.pivot(index='year', columns='nationality', values='avg_price')

fig, ax = plt.subplots(figsize=(12, 5))

if 'Indian' in nat_pivot.columns:
    ax.plot(nat_pivot.index, nat_pivot['Indian'], 'o-', label='Indian', color='orange', linewidth=2)
if 'Overseas' in nat_pivot.columns:
    ax.plot(nat_pivot.index, nat_pivot['Overseas'], 's--', label='Overseas', color='blue', linewidth=2)

ax.set_xlabel('Year')
ax.set_ylabel('Average Price (₹ Crore, 2024 terms)')
ax.set_title('Average Auction Price: Indian vs Overseas Players')
ax.legend()
ax.set_xticks(nat_pivot.index)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/analysis/fig_nationality_premium.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nOverall nationality comparison:")
print(df.groupby('nationality')['price_2024_cr'].agg(['mean', 'median', 'count']).round(2))

## 6. Performance vs Price Scatterplots

In [ ]:
df_perf = df[df['runs'].notna() & df['price_2024_cr'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.scatter(df_perf['runs'], df_perf['price_2024_cr'], alpha=0.5, edgecolors='black', linewidth=0.5)
ax1.set_xlabel('Runs Scored (same season)')
ax1.set_ylabel('Price (₹ Crore, 2024 terms)')
ax1.set_title('Runs vs Auction Price')

z = np.polyfit(df_perf['runs'], df_perf['price_2024_cr'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_perf['runs'].min(), df_perf['runs'].max(), 100)
ax1.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Linear fit')
ax1.legend()

ax2 = axes[1]
df_bowl = df_perf[df_perf['wickets'] > 0]
ax2.scatter(df_bowl['wickets'], df_bowl['price_2024_cr'], alpha=0.5, edgecolors='black', linewidth=0.5, color='green')
ax2.set_xlabel('Wickets Taken (same season)')
ax2.set_ylabel('Price (₹ Crore, 2024 terms)')
ax2.set_title('Wickets vs Auction Price')

z2 = np.polyfit(df_bowl['wickets'], df_bowl['price_2024_cr'], 1)
p2 = np.poly1d(z2)
x_line2 = np.linspace(df_bowl['wickets'].min(), df_bowl['wickets'].max(), 100)
ax2.plot(x_line2, p2(x_line2), 'r--', linewidth=2, label=f'Linear fit')
ax2.legend()

plt.tight_layout()
plt.savefig('../data/analysis/fig_performance_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

corr_runs = df_perf['runs'].corr(df_perf['price_2024_cr'])
corr_wickets = df_bowl['wickets'].corr(df_bowl['price_2024_cr'])
print(f"\nCorrelation (runs vs price): {corr_runs:.3f}")
print(f"Correlation (wickets vs price): {corr_wickets:.3f}")

## 7. Top All-Time Deals (Inflation-Adjusted)

In [ ]:
top_20 = df.nlargest(20, 'price_2024_cr')[[
    'year', 'player_name', 'team_x', 'role', 'nationality',
    'price_nominal_cr', 'price_2024_cr'
]].copy()
top_20['rank'] = range(1, 21)
top_20 = top_20[['rank', 'year', 'player_name', 'team_x', 'role', 'price_nominal_cr', 'price_2024_cr']]

print("Top 20 All-Time IPL Deals (Inflation-Adjusted to 2024 ₹):")
print(top_20.to_string(index=False))

## Summary Statistics

In [ ]:
print("=== Dataset Summary ===")
print(f"Total auction records: {len(df)}")
print(f"Years covered: {df['year'].min()} - {df['year'].max()}")
print(f"Unique players: {df['player_name'].nunique()}")
print(f"\nRecords with performance data: {df['runs'].notna().sum()} ({df['runs'].notna().mean()*100:.1f}%)")
print(f"\nTotal nominal spending: ₹{df['price_nominal_cr'].sum():.0f} Cr")
print(f"Total real spending (2024 ₹): ₹{df['price_2024_cr'].sum():.0f} Cr")
print(f"\nMean price (nominal): ₹{df['price_nominal_cr'].mean():.2f} Cr")
print(f"Mean price (2024 ₹): ₹{df['price_2024_cr'].mean():.2f} Cr")
print(f"Median price (2024 ₹): ₹{df['price_2024_cr'].median():.2f} Cr")